# ST-OMR Meter V5-2 — 1200 TRAIN Full-Meter BBox

**Canonical continuation:** V5-1’de kabul edilen 30 kırmızı full-meter BBox değişmeden korunur.
Kalan 1170 TRAIN örneğinde aynı kural devam eder: tek kutu üst+alt meter rakamlarını birlikte kapsar.
**BBox ikiye bölünmez; numerator/denominator kutusu üretilmez.** VAL ve FINAL HOLDOUT kapalıdır.

**İzleme kuralı:** Uzun sürebilecek kurulum, precheck ve audit işleri arka plan worker içinde çalışır.
Her worker hücresinde canlı izleme ekranı, geçen süre, faz ve güvenlik bayrakları görünür.


In [ ]:
from google.colab import drive
from IPython.display import HTML, display
display(HTML("""
<div style='padding:10px;border:1px solid #999;border-radius:8px;background:#fafafa'>
<b>İZLEME — DRIVE MOUNT</b><br>
Durum: kullanıcı yetkilendirmesi bekleniyor.<br>
Bu hücre etkileşimlidir; arka plana alınmaz.
</div>
"""))
drive.mount('/content/drive')
display(HTML("<div style='padding:8px;border:1px solid #999;border-radius:8px'><b>DRIVE MOUNT: PASS</b></div>"))


In [ ]:
import os, pathlib, subprocess, sys, shutil, threading, time, html
from IPython.display import HTML, display, clear_output

EXPECTED_CODE_SHA = 'de46b6c163376a0bab6c6ac768bca6af76c7afa5'
BRANCH = 'agent/meter-v5-2-train-bbox-scale-contract'
REPO_URL = 'https://github.com/khfy7wpr5p-maker/st-omr-training.git'
REPO_DIR = pathlib.Path('/content/st-omr-training-v5-2')

def _monitor_panel(title, phase, elapsed, status='RUNNING', detail='', safety=True):
    safety_text = (
        'FINAL_HOLDOUT=LOCKED | TRAINING=CLOSED | MODEL=CLOSED | INFERENCE=0'
        if safety else ''
    )
    return HTML(f"""
    <div style='font-family:Arial,sans-serif;padding:12px;border:1px solid #888;border-radius:9px;background:#fafafa'>
      <div style='font-size:17px;font-weight:bold'>{html.escape(title)}</div>
      <div><b>Durum:</b> {html.escape(str(status))}</div>
      <div><b>Faz:</b> {html.escape(str(phase))}</div>
      <div><b>Geçen süre:</b> {int(elapsed)} sn</div>
      <div><b>Detay:</b> {html.escape(str(detail))}</div>
      <div style='margin-top:6px'><b>Güvenlik:</b> {html.escape(safety_text)}</div>
    </div>
    """)

def run_monitored_background(title, worker, *, heartbeat=5, state=None, safety=True):
    state = state or {'phase':'starting', 'detail':'', 'result':None, 'error':None}
    def _wrapped():
        try:
            state['result'] = worker(state)
            state['phase'] = 'done'
        except BaseException as exc:
            state['error'] = exc
            state['phase'] = 'error'
            state['detail'] = repr(exc)
    t = threading.Thread(target=_wrapped, name=title.replace(' ','-').lower(), daemon=False)
    started = time.monotonic()
    t.start()
    while t.is_alive():
        clear_output(wait=True)
        display(_monitor_panel(
            title, state.get('phase','running'), time.monotonic()-started,
            'RUNNING', state.get('detail',''), safety=safety
        ))
        t.join(timeout=heartbeat)
    t.join()
    elapsed = time.monotonic()-started
    clear_output(wait=True)
    if state['error'] is not None:
        display(_monitor_panel(title, state['phase'], elapsed, 'FAIL-CLOSED', state.get('detail',''), safety=safety))
        raise state['error']
    display(_monitor_panel(title, state['phase'], elapsed, 'PASS', state.get('detail',''), safety=safety))
    return state['result']

def _setup_worker(state):
    state['phase'] = 'repo_cleanup'
    state['detail'] = str(REPO_DIR)
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)

    state['phase'] = 'git_clone'
    state['detail'] = BRANCH
    subprocess.run(['git','clone','--depth','30','--branch',BRANCH,REPO_URL,str(REPO_DIR)], check=True)

    state['phase'] = 'sha_pin'
    state['detail'] = EXPECTED_CODE_SHA
    subprocess.run(['git','-C',str(REPO_DIR),'checkout','--detach',EXPECTED_CODE_SHA], check=True)
    actual = subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'], text=True).strip()
    assert actual == EXPECTED_CODE_SHA, (actual, EXPECTED_CODE_SHA)

    state['phase'] = 'dependency_install'
    state['detail'] = 'Pillow==12.3.0'
    subprocess.run([sys.executable,'-m','pip','install','-q','Pillow==12.3.0'], check=True)

    state['phase'] = 'python_path'
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    state['detail'] = f'CODE_PIN={actual}'
    return actual

ACTUAL_CODE_SHA = run_monitored_background(
    'V5-2 KURULUM İZLEME',
    _setup_worker,
    heartbeat=5,
    safety=True,
)
print('CODE_PIN=PASS', ACTUAL_CODE_SHA)


## Güvenli precheck — mevcut 30 seed + 1200 TRAIN bağlama

Bu hücre exact clean dataset’i, V5-1 seed hash’lerini ve 1200 TRAIN image binding’ini doğrular.
İlk 30 kabul edilmiş BBox değiştirilmeden yeni 1200 kayıt dosyasına seed edilir.
Uzun dosya doğrulamaları **arka plan worker** içinde çalışır; izleme ekranı 5 saniyede bir güncellenir.
Model/training/inference çalışmaz.


In [ ]:
import pathlib
from st_omr_training.meter_v5_2_train_bbox_scale import ScaleAnnotationSession, TRAIN_TOTAL

MYDRIVE = pathlib.Path('/content/drive/MyDrive')
DATA_ROOT = MYDRIVE / 'TEST' / 'METER_V2_1500_PACKAGE_AB_CLEAN'
if not MYDRIVE.is_dir():
    raise RuntimeError(f'MyDrive not mounted: {MYDRIVE}')
if not DATA_ROOT.is_dir():
    raise RuntimeError(f'Authoritative dataset not found: {DATA_ROOT}')

def _precheck_worker(state):
    state['phase'] = 'construct_scale_session'
    state['detail'] = 'seed hashes + 1200 TRAIN image binding'
    session = ScaleAnnotationSession(data_root=DATA_ROOT)
    state['phase'] = 'validate_counts'
    state['detail'] = f'handled={session.handled_count} pass={session.pass_count} review={session.review_count}'
    assert len(session.samples) == TRAIN_TOTAL == 1200
    assert session.handled_count >= 30
    assert session.pass_count >= 30
    assert session.resume_index() >= 30
    return session

SESSION = run_monitored_background(
    'V5-2 PRECHECK İZLEME',
    _precheck_worker,
    heartbeat=5,
    safety=True,
)
print('PRECHECK=PASS')
print('handled=', SESSION.handled_count, 'pass=', SESSION.pass_count, 'review=', SESSION.review_count)
print('resume_index=', SESSION.resume_index())
print('FIRST_30_SEEDS=LOCKED; BBOX_SPLIT=False; FINAL_HOLDOUT=LOCKED; TRAINING=False; MODEL_OPENED=False; INFERENCE_COUNT=0')


## 1200 TRAIN BBox arayüzü

Bu hücre uzun bir batch işi çalıştırmaz; insan etkileşimli annotation ekranıdır.
Arayüzün kendisi sürekli izleme ekranıdır: PASS, REVIEW, kalan işlenmemiş örnek ve örnek numarası canlı görünür.
İlk 30 seed kilitlidir. Her yeni örnekte yalnız **tek full-meter kırmızı kutu** çizilir.


In [ ]:
from st_omr_training.meter_v5_2_train_bbox_colab import launch_colab_scale

display(_monitor_panel(
    'V5-2 ANNOTATION İZLEME',
    'interactive_ui',
    0,
    'READY',
    f'resume_index={SESSION.resume_index()} | handled={SESSION.handled_count} | pass={SESSION.pass_count} | review={SESSION.review_count}',
    safety=True,
))
SESSION = launch_colab_scale(data_root=str(DATA_ROOT), session=SESSION)
print('V5_2_UI=READY')
print('resume_index=', SESSION.resume_index())
print('handled=', SESSION.handled_count, 'pass=', SESSION.pass_count, 'review=', SESSION.review_count)
print('FINAL_HOLDOUT=LOCKED; TRAINING=False; MODEL_OPENED=False; INFERENCE_COUNT=0')


## Mekanik audit — 1200/1200 tamamlandıktan sonra

Audit insan BBox’larını değiştirmez. 1200 PASS, 400/class, seed mutation=0 ve diğer mekanik kapıları doğrular.
Uzun sürebileceği için **arka plan worker** içinde çalışır ve canlı audit izleme ekranı gösterir.
PASS olsa bile training otomatik açılmaz; ayrıca insan görsel QA gerekir.


In [ ]:
import json
from st_omr_training.meter_v5_2_train_bbox_scale import write_train_audit

def _audit_worker(state):
    state['phase'] = 'write_train_audit'
    state['detail'] = '1200 TRAIN kayıt mekanik doğrulaması'
    audit_path = write_train_audit(DATA_ROOT)
    state['phase'] = 'read_audit'
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    state['detail'] = (
        f"mechanical_gate={audit.get('mechanical_gate')} | "
        f"pass={audit.get('pass_count')} | review={audit.get('review_count')}"
    )
    return audit_path, audit

AUDIT_PATH, AUDIT = run_monitored_background(
    'V5-2 AUDIT İZLEME',
    _audit_worker,
    heartbeat=5,
    safety=True,
)
print(json.dumps(AUDIT, indent=2, sort_keys=True))
print('AUDIT_PATH=', AUDIT_PATH)
print('TRAINING_AUTHORIZED=', AUDIT['training_authorized'])
print('HUMAN_VISUAL_REVIEW_REQUIRED=', AUDIT['human_visual_review_required'])
